# Base de connaissances d'un débat : propositions, arguments, conflits — jumeau C#

Jumeau C# de [04b-Knowledge-Base-Python](./Argumentation-04b-Knowledge-Base-Python.ipynb) :
même modèle (propositions + arguments), même sémantique lexicale — population
transitive, négation par préfixe `¬`, `entails` comme test d'appartenance,
écrasement par contenu — portée ici en **C# BCL natif** (pas de NuGet, pas de
Tweety : l'organe du carnet est le `knowledge_base.py` du dépôt, et son port
tient dans les collections BCL).

Les trois limites mesurées du jumeau Python (pas d'explosion, la double négation
`¬¬P` inconnue, l'écrasement par contenu) sont reproduites à l'identique : c'est
le comportement du tronc, **documenté et pas corrigé**.

**Série Argumentation** | | Précédent : [04-Dialogues-Protocolises-Python](./Argumentation-04-Dialogues-Protocolises-Python.ipynb) | Suivant : [05-Formal-Verification-Python](./Argumentation-05-Formal-Verification-Python.ipynb) | [↑ Série Argumentation](README.md) | [jumeau Python](Argumentation-04b-Knowledge-Base-Python.ipynb)

## §1 — Le modèle : propositions et arguments

Une base de connaissances de débat tient en deux registres :

1. **les propositions** — des contenus de parole indexés par leur texte ;
2. **les arguments** — des structures `[prémisses] -> conclusion`, qui
   *portent* leurs propositions : ajouter un argument enregistre
   automatiquement ses prémisses **et** sa conclusion.

C'est ce second point qui fait la base : la population de propositions est
**transitive** — on n'ajoute que des arguments, et les propositions
apparaissent toutes seules. Le banc de test est le même que le jumeau Python :
un débat à deux camps sur le télétravail et la concentration.


In [1]:
// --- Port BCL de knowledge_base.py : le comptoir lexicale du debat ---
// Sémantique portée a l'identique : population transitive, negation par
// prefixe "¬", entails = test d'appartenance, ecrasement par contenu.

using System;
using System.Collections.Generic;
using System.Linq;

// Helper d'affichage : en headless, Console.WriteLine peut etre avale.
static void Show(object o)
{
    try { display(o); } catch { Console.WriteLine(o); }
}

// Garde de temoin (equivalent de l'assert du jumeau Python) : passe dans
// l'execution normale, s'arrete a la premiere divergence.
static void AssertTrue(bool cond, string msg)
{
    if (!cond) throw new Exception($"Temoin non tenu : {msg}");
}

class Proposition
{
    public string Content { get; }
    public bool? TruthValue { get; }
    public double Confidence { get; }
    public string? Source { get; }

    public Proposition(string content, bool? truthValue = null,
        double confidence = 1.0, string? source = null)
    { Content = content; TruthValue = truthValue; Confidence = confidence; Source = source; }

    // L'egalite et le hachage ignorent truthValue, confidence et source :
    // deux propositions de meme contenu sont la meme proposition (divergence #4).
    public override int GetHashCode() => Content.GetHashCode();
    public override bool Equals(object? o) => o is Proposition p && p.Content == Content;
    public override string ToString() => Content;
}

class FormalArgument
{
    public List<Proposition> Premises { get; }
    public Proposition Conclusion { get; }
    public double Strength { get; }
    public string? Scheme { get; }
    public string Id { get; }

    public FormalArgument(List<Proposition> premises, Proposition conclusion,
        double strength = 1.0, string? scheme = null)
    {
        Premises = premises; Conclusion = conclusion; Strength = strength; Scheme = scheme;
        Id = Guid.NewGuid().ToString();
    }

    public override string ToString() =>
        $"[{string.Join(", ", Premises.Select(p => (object)p))}] -> {Conclusion}";
}

static Proposition Negation(Proposition prop) => new("¬" + prop.Content);

class KnowledgeBase
{
    public Dictionary<string, Proposition> Propositions { get; } = new();
    public Dictionary<string, FormalArgument> Arguments { get; } = new();

    public void AddProposition(Proposition prop) => Propositions[prop.Content] = prop;

    public void AddArgument(FormalArgument arg)
    {
        Arguments[arg.Id] = arg;
        foreach (var premise in arg.Premises) AddProposition(premise);
        AddProposition(arg.Conclusion);
    }

    public List<FormalArgument> FindSupportingArguments(Proposition prop) =>
        Arguments.Values.Where(a => a.Conclusion.Content == prop.Content).ToList();

    public List<FormalArgument> FindAttackingArguments(Proposition prop) =>
        Arguments.Values.Where(a => a.Conclusion.Content == "¬" + prop.Content).ToList();

    public bool IsConsistent()
    {
        foreach (var content in Propositions.Keys)
            if (Propositions.ContainsKey("¬" + content)) return false;
        return true;
    }

    public bool Entails(Proposition prop) => Propositions.ContainsKey(prop.Content);

    public List<Proposition> GetAllPropositions() => Propositions.Values.ToList();
    public List<FormalArgument> GetAllArguments() => Arguments.Values.ToList();
}

The below script needs to be able to find the current output cell; this is an easy method to get it.


(27,18): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(30,40): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(36,39): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(45,18): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(49,38): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



In [2]:
// Le banc a deux camps (identique au jumeau Python).
KnowledgeBase ConstruireDeuxCamps()
{
    var kb = new KnowledgeBase();
    kb.AddArgument(new FormalArgument(
        premises: new() { new Proposition("Le télétravail isole les collaborateurs") },
        conclusion: new Proposition("Le télétravail réduit la concentration"),
        scheme: "cause_effect"));
    kb.AddArgument(new FormalArgument(
        premises: new() { new Proposition("Les pauses régulières restaurent la concentration") },
        conclusion: new Proposition("¬Le télétravail réduit la concentration")));
    return kb;
}

KnowledgeBase ConstruireUnSeulCamp()
{
    var kb = new KnowledgeBase();
    kb.AddArgument(new FormalArgument(
        premises: new() { new Proposition("Le télétravail isole les collaborateurs") },
        conclusion: new Proposition("Le télétravail réduit la concentration"),
        scheme: "cause_effect"));
    return kb;
}

var kb = ConstruireDeuxCamps();
var kbSaine = ConstruireUnSeulCamp();

Show($"Propositions enregistrées ({kb.GetAllPropositions().Count}) :");
foreach (var p in kb.GetAllPropositions()) Show($"  - {p.Content}");
Show($"Arguments enregistrés : {kb.GetAllArguments().Count}");

Propositions enregistrées (4) :

  - Le télétravail isole les collaborateurs

  - Le télétravail réduit la concentration

  - Les pauses régulières restaurent la concentration

  - ¬Le télétravail réduit la concentration

Arguments enregistrés : 2

### Lecture : la population transitive

Quatre propositions, deux arguments — alors qu'on n'a ajouté que deux
arguments. La prémisse de chaque camp (« Le télétravail isole les
collaborateurs », « Les pauses régulières restaurent la concentration »)
n'a été posée par personne : c'est son argument qui l'a portée. La base
est un *comptoir* de tout ce que le débat a dit, pas une collection qu'on
alimente à la main.


In [3]:
var premisse = new Proposition("Le télétravail isole les collaborateurs");
Show($"entails(premisse) = {kb.Entails(premisse)}");
Show("'Le télétravail isole les collaborateurs' n'a jamais été ajoutée explicitement —");
Show("c'est l'argument qui la porte : la population est transitive.");
AssertTrue(kb.Entails(premisse), "entails(premisse) attendu True");

var intrus = new Proposition("Le télétravail augmente la productivité");
Show($"entails(intrus) = {kb.Entails(intrus)}");
AssertTrue(!kb.Entails(intrus), "entails(intrus) attendu False");

entails(premisse) = True

'Le télétravail isole les collaborateurs' n'a jamais été ajoutée explicitement —

c'est l'argument qui la porte : la population est transitive.

entails(intrus) = False

### Lecture : appartient, sans inférer

`entails(premisse)` rend `True` sans que la prémisse ait jamais été ajoutée
explicitement : l'appartenance est une conséquence de l'argument qui la
porte. Et `entails(intrus)` rend `False` pour une proposition jamais dite —
le nom `entails` promet « implique », la sémantique est « contient » :
c'est la divergence #2 du port, mesurée ici.


## §2 — Support et attaque : la sémantique lexicale

Le carnet ne raisonne pas sur les formules : il *lit* les contenus. Le
**support** d'une proposition, ce sont les arguments dont la conclusion
est *exactement* ce contenu ; l'**attaque**, ce sont ceux dont la conclusion
est le contenu préfixé de `¬` — la négation n'est pas un opérateur logique,
c'est une construction de chaîne.


In [4]:
var prop = new Proposition("Le télétravail réduit la concentration");
var soutiens = kb.FindSupportingArguments(prop);
var attaques = kb.FindAttackingArguments(prop);
Show($"Soutiens de « {prop.Content} » : {soutiens.Count}");
foreach (var a in soutiens) Show($"  {a}  (schéma : {a.Scheme ?? "None"})");
Show($"Attaques (conclusion == negation(prop) == « {Negation(prop).Content} ») : {attaques.Count}");
foreach (var a in attaques) Show($"  {a}  (schéma : {a.Scheme ?? "None"})");
AssertTrue(soutiens.Count == 1 && attaques.Count == 1, "1 soutien + 1 attaque attendus");

Soutiens de « Le télétravail réduit la concentration » : 1

  [Le télétravail isole les collaborateurs] -> Le télétravail réduit la concentration  (schéma : cause_effect)

Attaques (conclusion == negation(prop) == « ¬Le télétravail réduit la concentration ») : 1

  [Les pauses régulières restaurent la concentration] -> ¬Le télétravail réduit la concentration  (schéma : None)

### Lecture : une attaque, par le texte

L'argument du second camp conclut exactement à
`¬Le télétravail réduit la concentration` — le préfixe `¬` posé sur le
contenu du premier. La base le reconnaît comme attaque par une comparaison
de chaînes : un caractère de plus ou de moins et l'attaque serait invisible.
C'est le prix de la simplicité lexicale, et son domaine : des contenus
normalisés, pas des formules.


In [5]:
var prop2 = new Proposition("Le télétravail réduit la concentration");
var soutiens2 = kbSaine.FindSupportingArguments(prop2);
var attaques2 = kbSaine.FindAttackingArguments(prop2);
Show($"Base saine — soutiens : {soutiens2.Count}, attaques : {attaques2.Count}");
AssertTrue(soutiens2.Count == 1 && attaques2.Count == 0, "1 soutien + 0 attaque attendus");

Base saine — soutiens : 1, attaques : 0

### Lecture : la base saine, témoin

Même argument, second camp retiré : 1 soutien, 0 attaque. Le contraste
avec la base à deux camps (1+1) isole exactement l'effet de l'argument
attaquant : rien d'autre ne change.


## §2 bis — Ce que la base ne sait pas dire : le graphe des réponses

Le champ `argument_network` du tronc porte ce que la base ne porte pas :
qui *répond à qui*. La base du carnet répond « quels arguments soutiennent
P / attaquent P » ; le réseau répond « quel argument a été *posé pour
répondre* à quel autre » — une relation entre arguments, pas entre
argument et proposition.


In [6]:
// §2 bis — Le champ argument_network du tronc : qui repond a qui.
// La base du carnet ne le porte pas (mesure du §2) ; on le construit ici
// pour montrer ce qu'elle ne sait pas dire.
var arguments = new Dictionary<string, string>
{
    ["A_teletravail"] = "Le teletravail reduit la concentration des equipes",
    ["B_rythme"] = "Le rythme des jours equipe fixe la concentration, pas le lieu",
    ["C_correctif"] = "Les journees equipe hebdomadaires restaurent la concentration",
};

// argument_network : la liste de X = les arguments a qui X repond.
var argumentNetwork = new Dictionary<string, List<string>>
{
    ["B_rythme"] = new() { "A_teletravail" },
    ["C_correctif"] = new() { "A_teletravail" },
    ["A_teletravail"] = new(),
};

var cible = "A_teletravail";
var repondants = argumentNetwork
    .Where(kv => kv.Value.Contains(cible)).Select(kv => kv.Key).ToList();
Show($"Qui repond a {cible} ? -> {string.Join(", ", repondants)}");
Show("Degre sortant : " + string.Join(", ",
    argumentNetwork.Select(kv => $"{kv.Key} -> {kv.Value.Count}")));

Qui repond a A_teletravail ? -> B_rythme, C_correctif

Degre sortant : B_rythme -> 1, C_correctif -> 1, A_teletravail -> 0

### Lecture : le même débat, une géométrie de plus

`B_rythme` et `C_correctif` sont des *réponses* à `A_teletravail` : le
degré sortant de A vaut 0, ceux de B et C valent 1. La base du §2 ne
pouvait pas exprimer ce qui précède — elle ne voit que les conclusions,
pas la topologie de l'échange. Le carnet mesure l'écart au lieu de le
combler : c'est la divergence portée en documentation.


## §3 — Cohérence et appartenance : les deux questions du débat

Deux questions se posent sur une base, et répondent de deux façons
différentes. **La base est-elle cohérente ?** — `is_consistent()` rend
`False` dès qu'une proposition et sa négation lexicale cohabitent : c'est
le marqueur qu'un *débat a lieu* sur cette proposition, pas un dérapage du
registre. **La base contient-elle P ?** — `entails`, le test
d'appartenance du §1, sans aucune inférence.


In [7]:
Show($"deux_camps   is_consistent -> {kb.IsConsistent()}");
Show($"un_seul_camp is_consistent -> {kbSaine.IsConsistent()}");
AssertTrue(!kb.IsConsistent() && kbSaine.IsConsistent(), "incohérente / cohérente attendues");

var queries = new (string contenu, bool attendu)[]
{
    ("Le télétravail isole les collaborateurs", true),
    ("Le télétravail augmente la productivité", false),
    ("¬Le télétravail réduit la concentration", true),
};
foreach (var (contenu, attendu) in queries)
{
    var got = kb.Entails(new Proposition(contenu));
    Show($"entails({contenu}) = {got}   (attendu {attendu})");
    AssertTrue(got == attendu, $"entails({contenu}) attendu {attendu}");
}

deux_camps   is_consistent -> False

un_seul_camp is_consistent -> True

entails(Le télétravail isole les collaborateurs) = True   (attendu True)

entails(Le télétravail augmente la productivité) = False   (attendu False)

entails(¬Le télétravail réduit la concentration) = True   (attendu True)

### Lecture : incohérence = débat ouvert

La base à deux camps est incohérente *par définition* : les deux camps y
ont laissé leurs conclusions contradictoires. Retirer le second camp, et
la cohérence revient — sans que rien d'autre ne change. Les trois
requêtes `entails` de la cellule précédente y puisent leur valeur :
l'appartenance y est indépendante de la cohérence.


## §4 — Trois limites mesurées, documentées plutôt que corrigées

Le port conserve le comportement du tronc — et ses angles morts. Chacun
est mesuré sur une base construite pour le révéler, et chacun dit ce
qu'une base lexicale n'est **pas**.


In [8]:
// Limite 1 : pas d'explosion. Logique classique : d'une contradiction
// tout s'ensuit. Ici, la base est incoherente et entails(arbitraire)
// reste faux.
var q = new Proposition("Une proposition quelconque jamais posée");
Show($"base incohérente : is_consistent = {kb.IsConsistent()}");
Show($"entails(arbitraire) = {kb.Entails(q)}   <- pas d'ex falso quodlibet");
AssertTrue(!kb.Entails(q), "pas d'explosion attendue");

var kbMin = new KnowledgeBase();
kbMin.AddProposition(new Proposition("A"));
Show($"base = {{A}} : entails(B) = {kbMin.Entails(new Proposition("B"))}");
AssertTrue(!kbMin.Entails(new Proposition("B")), "B non posée");

base incohérente : is_consistent = False

entails(arbitraire) = False   <- pas d'ex falso quodlibet

base = {A} : entails(B) = False

### Limite 1 — pas d'explosion

En logique classique, d'une contradiction tout s'ensuit (principe
d'explosion). Ici, la base incohérente ne *débouche* sur rien :
`entails` sur une proposition arbitraire reste `False`. Le « tout
s'ensuit » n'a pas de lieu : l'appartenance n'infère pas. L'incohérence
est un état du registre, pas une source d'inférences.


In [9]:
// Limite 2 : la double negation n'existe pas. Negation(Negation(P)) = "¬¬P",
// pas "P".
var p = new Proposition("Le débat est ouvert");
var doubleNeg = Negation(Negation(p));
Show($"P                     = {p.Content}");
Show($"negation(negation(P)) = {doubleNeg.Content}");

var kbDn = new KnowledgeBase();
kbDn.AddProposition(p);
kbDn.AddProposition(doubleNeg);
Show($"Base {{P, ¬¬P}} : is_consistent = {kbDn.IsConsistent()}   <- aucune paire P/¬P exacte");
AssertTrue(kbDn.IsConsistent(), "P et ¬¬P cohabitent sans conflit");
AssertTrue(kbDn.Entails(p) && kbDn.Entails(doubleNeg), "les deux propositions sont présentes");

P                     = Le débat est ouvert

negation(negation(P)) = ¬¬Le débat est ouvert

Base {P, ¬¬P} : is_consistent = True   <- aucune paire P/¬P exacte

### Limite 2 — la double négation inconnue

`Negation(Negation(p))` produit la chaîne `¬¬P` — une proposition
*distincte* de `P`. Une base contenant `P` et `¬¬P` est déclarée
cohérente : le test de cohérence cherche la paire exacte `P` / `¬P`, et
`¬¬P` n'est pas `¬P`. La négation est lexicale, point : il n'y a nulle
part de loi de double négation, et rien ne devrait en attendre.


In [10]:
// Limite 3 : l'ecrasement par contenu. Deux Proposition de meme contenu,
// un seul exemplaire — le dernier ajoute.
var kbEc = new KnowledgeBase();
kbEc.AddProposition(new Proposition("La réunion est utile", confidence: 0.9, source: "expert"));
kbEc.AddProposition(new Proposition("La réunion est utile", confidence: 0.1, source: "sondage"));
var restantes = kbEc.GetAllPropositions();
Show($"Propositions en base : {restantes.Count}");
foreach (var propX in restantes)
    Show($"  {propX.Content}  confidence={propX.Confidence}  source={propX.Source}");
AssertTrue(restantes.Count == 1 && restantes[0].Confidence == 0.1, "écrasement par le dernier arrivant");

// L'egalite STRUCTURELLE l'explique : deux Proposition de meme contenu
// sont egales, quelle que soit leur confiance.
var a = new Proposition("P", confidence: 0.9);
var b = new Proposition("P", confidence: 0.2);
Show($"Proposition('P', 0.9) == Proposition('P', 0.2) : {a.Equals(b)}");
Show($"hash égaux : {a.GetHashCode() == b.GetHashCode()}");

Propositions en base : 1

  La réunion est utile  confidence=0,1  source=sondage

Proposition('P', 0.9) == Proposition('P', 0.2) : True

hash égaux : True

### Limite 3 — l'écrasement par contenu

Le registre est indexé par contenu : deux propositions identiques par le
texte ne laissent qu'un exemplaire — le dernier ajouté, quelle que soit sa
confiance. Le sondage à 0,1 écrase l'expert à 0,9. L'égalité structurelle
l'explique : deux `Proposition` de même contenu sont *égales* (et de même
hachage), quoi qu'il en soit de leur confiance — c'est la divergence #4
du port, et c'est ce qui rend l'écrasement inévitable.


## Vérifiez votre compréhension

Trois exercices, un par section du carnet. Les stubs s'exécutent sans
erreur (règle C.1) et la méthode de chacun est déjà démontrée dans le
carnet :

- **Exercice 1** → §2 : la base du banc a 1 soutien et 1 attaque ; il faut
  composer une base qui en ait 2 et 0.
- **Exercice 2** → §3 : `IsConsistent` est `True` sur le banc ; un *seul*
  argument doit suffire à le rendre `False`.
- **Exercice 3** → §4 : prédire avant d'exécuter — la double négation ne
  fait ni incohérence ni appartenance.


## Exercice 1 — deux soutiens, zéro attaque

Composer `maBase` pour que `FindSupportingArguments(cibleX)` rende
exactement **2** arguments et `FindAttackingArguments(cibleX)` en rende
**0**.

Indications : deux arguments doivent conclure *exactement* à `cibleX` —
et aucun ne doit conclure à `¬` + ce contenu. La sémantique lexicale du
§2 fait le reste : c'est le texte exact des conclusions qui compte.

Témoin : les compteurs ci-dessous doivent passer à 2 et 0.


In [11]:
// Exercice 1 : composer maBase pour 2 soutiens et 0 attaque sur cibleX.
var maBase = new KnowledgeBase();
// TODO etudiant : ajouter deux arguments concluant exactement a cibleX
// (aucun ne doit conclure a sa negation lexicale).
var cibleX = new Proposition("Le tramway réduit les embouteillages");

int nbSoutiens = 0; // TODO etudiant : maBase.FindSupportingArguments(cibleX).Count
int nbAttaques = 0; // TODO etudiant : maBase.FindAttackingArguments(cibleX).Count
Show($"Exercice a completer — soutiens : {nbSoutiens} (attendus 2), attaques : {nbAttaques} (attendues 0)");

Exercice a completer — soutiens : 0 (attendus 2), attaques : 0 (attendues 0)

## Exercice 2 — un seul argument pour casser la cohérence

La base pose d'abord la proposition `Q` seule (cohérente). Ajouter **un
seul** argument à `kbDefi` pour que `IsConsistent()` devienne `False`.

Indications : l'argument doit conclure à la *négation lexicale* de Q —
la chaîne `¬` + contenu de Q — et la cohérence du §3 fait le reste :
l'argument apporte sa conclusion, la population transitive s'occupe du
reste.

Témoin : le verdict affiché doit passer de `True` à `False`.


In [12]:
// Exercice 2 : un SEUL AddArgument doit faire basculer IsConsistent() a false.
var kbDefi = new KnowledgeBase();
kbDefi.AddProposition(new Proposition("La formation est utile"));
// TODO etudiant : ajouter un argument dont la conclusion est la negation
// lexicale ("¬" + contenu) de la proposition posee ci-dessus.
var verdict = kbDefi.IsConsistent();
Show($"Exercice a completer — base cohérente : {verdict} (attendu : false)");

Exercice a completer — base cohérente : True (attendu : false)

## Exercice 3 — prédire, puis vérifier

Avant d'exécuter la cellule de vérification, écrire dans `predictions`
les deux valeurs attendues :

1. `is_consistent` sur une base `{P, ¬¬P}` ;
2. `entails(¬P)` sur cette même base (attention : `¬P`, pas `¬¬P`).

Exécuter, puis comparer. La limite 2 du §4 explique la première ; la
seconde relève de l'appartenance stricte — `¬P` n'y figure que si
quelqu'un l'a posée.

Témoin : les deux valeurs réelles s'affichent sous les prédictions.


In [13]:
// Exercice 3 : predire, puis verifier.
var kbPredit = new KnowledgeBase();
kbPredit.AddProposition(new Proposition("Le débat est vif"));
kbPredit.AddProposition(new Proposition("¬¬Le débat est vif"));

var predictions = new List<string>();
// TODO etudiant : pousser vos deux predictions, par ex. :
// predictions.Add("is_consistent = ...");
// predictions.Add("entails(¬P) = ...");

Show($"Vos prédictions : {string.Join(" | ", predictions)}");
Console.WriteLine();
Show($"Réel : is_consistent = {kbPredit.IsConsistent()}");
Show($"Réel : entails(¬P) = {kbPredit.Entails(new Proposition("¬Le débat est vif"))}");

Vos prédictions : 

Réel : is_consistent = True

Réel : entails(¬P) = False

## À retenir

1. **La population est transitive.** On ajoute des arguments ; les
   propositions (prémisses et conclusions) apparaissent toutes seules —
   mesuré : 4 propositions pour 2 arguments, et `entails(premisse)` `True`
   sans ajout explicite.
2. **La sémantique est lexicale.** Support et attaque sont des comparaisons
   de chaînes ; la négation est un préfixe `¬`, pas un opérateur — mesuré :
   1 soutien + 1 attaque sur le banc, `¬¬P` distincte de `P`, incohérence
   dès la paire exacte `P` / `¬P`.
3. **Ce que la base ne dit pas est mesuré, pas tu.** Pas d'explosion
   (`entails(arbitraire)` `False` sur base incohérente), double négation
   inconnue (`{P, ¬¬P}` cohérente), écrasement par contenu (le sondage
   0,1 remplace l'expert 0,9). Ce sont les limites d'un comptoir lexical —
   documentées, et à ne pas confondre avec celles d'un moteur
   d'inférence.


## Pour aller plus loin

- [04 — Dialogues protocolisés](Argumentation-04-Dialogues-Protocolises-Python.ipynb)
  (jumeau C# :
  [04-CSharp](Argumentation-04-Dialogues-Protocolises-CSharp.ipynb)) :
  les protocoles dans lesquels ces arguments circulent — et le dialogue
  de grounding qui tranche sur le vrai Tweety.
- [05 — Vérification formelle](Argumentation-05-Formal-Verification-Python.ipynb)
  : confier les verdicts à un solveur.
- [Index de la sous-série ArgumentAnalysis](README.md) — la place de la
  base dans le parcours.

**Statut et pérennité.** Le module `knowledge_base.py` du dépôt est un
port local du tronc EPITA (restauré G8 #1184) ; ce jumeau en est le port
BCL, fidèle aux quatre divergences documentées dans la docstring du module.
Le dialogue de grounding du carnet 04 (vrai Tweety) consomme ce type
d'arguments : la base les range, le protocole les pose, le reasoner tranche.

**Référence** — P. M. Dung (1994). *On the acceptability of arguments in
an argumentation structure*, Artificial Intelligence 77(1-2) — le cadre
dont le dialogue de grounding du carnet 04 exécute la sémantique.
